# Baseline Models for Russian NER: Natasha / Slovnet and GLiNER Multi

This notebook evaluates two external baseline models on the factRuEval‑2016 dataset:

- **Natasha / Slovnet** — a classical Russian NLP pipeline.
- **GLiNER Multi** — a modern multilingual NER model.

The goal is to establish reference performance before training RuModernBERT.  
We convert character‑level spans from both models into BIO labels aligned with tokens and compute strict span‑level F1 using seqeval.

In [ ]:
import numpy as np
import torch
from natasha import Doc, NewsEmbedding, NewsNERTagger, Segmenter
from gliner import GLiNER
from datasets import DatasetDict

## Natasha / Slovnet Baseline

Natasha produces NER spans at the character level.  
We convert them into BIO labels using token offsets from the dataset.

In [ ]:
# Natasha baseline.

segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

def natasha_ner_char_spans(text: str):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)
    spans = []
    for span in doc.spans:
        spans.append((span.start, span.stop, span.type))
    return spans

def run_natasha_on_dataset(ds_split, limit=None):
    y_true, y_pred = [], []
    n = len(ds_split) if limit is None else min(limit, len(ds_split))
    for i in range(n):
        example = ds_split[i]
        tokens, gold = factru_example_to_labels(example)
        text, offsets = tokens_to_text_and_offsets(tokens)
        spans = natasha_ner_char_spans(text)
        pred = char_spans_to_bio(tokens, offsets, spans, label_set=ENTITY_TYPES)
        y_true.append(gold)
        y_pred.append(pred)
    return y_true, y_pred

y_true_nat, y_pred_nat = run_natasha_on_dataset(ds_small["test"], limit=1000)

## GLiNER Multi Baseline

GLiNER predicts multilingual entity spans.  
We map its labels to PER / ORG / LOC and convert spans to BIO format.

In [ ]:
# GLiNER baseline.

gliner_model_id = "urchade/gliner_multi-v2.1"
gliner = GLiNER.from_pretrained(gliner_model_id).to(DEVICE)

def gliner_char_spans(text: str, threshold=0.35):
    ents = gliner.predict_entities(
        text, labels=["person", "organization", "location"], threshold=threshold
    )
    out = []
    for ent in ents:
        lab = ent["label"].lower()
        if lab.startswith("person"):
            ent_type = "PER"
        elif lab.startswith("organization"):
            ent_type = "ORG"
        elif lab.startswith("location"):
            ent_type = "LOC"
        else:
            continue
        out.append((ent["start"], ent["end"], ent_type))
    return out

def run_gliner_on_dataset(ds_split, limit=None, threshold=0.35):
    y_true, y_pred = [], []
    n = len(ds_split) if limit is None else min(limit, len(ds_split))
    for i in range(n):
        example = ds_split[i]
        tokens, gold = factru_example_to_labels(example)
        text, offsets = tokens_to_text_and_offsets(tokens)
        spans = gliner_char_spans(text, threshold=threshold)
        pred = char_spans_to_bio(tokens, offsets, spans, label_set=ENTITY_TYPES)
        y_true.append(gold)
        y_pred.append(pred)
    return y_true, y_pred

y_true_gl, y_pred_gl = run_gliner_on_dataset(ds_small["test"], limit=1000)

## Evaluation

We compute strict span‑level F1 using seqeval.  
This metric requires exact match of entity boundaries and types.

In [ ]:
# Evaluation.

res_nat = seqeval_strict_micro_f1(y_true_nat, y_pred_nat)
res_gl = seqeval_strict_micro_f1(y_true_gl, y_pred_gl)

res_nat, res_gl

## Summary

These baselines provide reference performance for Russian NER:

- Natasha / Slovnet — classical Russian NLP baseline  
- GLiNER Multi — modern multilingual NER baseline  

Their results will be compared against RuModernBERT fine‑tuning in the next notebook.